# Demonstracao do detector de imagens - Fase 3.9

Este notebook mostra a estimativa do modelo para imagens fornecidas pelo usuario. Nao treina, calibra nem mede acuracia. Execute 1-5, depois escolha 6 (navegador Colab) ou 7 (Drive/VS Code).


In [ ]:
# 1. Verifique PyTorch e dispositivo.
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', DEVICE, '| PyTorch:', torch.__version__)


In [ ]:
# 2. Monte o Drive do grupo; nenhum dataset e aberto.
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
RUN_DIR = ROOT / 'modelos/fase3_9_tgif_sd2fr_v1'
if not ROOT.is_dir(): raise FileNotFoundError('TCC_Final nao encontrado neste Drive')
print('Pasta do checkpoint:', RUN_DIR)


In [ ]:
# 3. Arquitetura identica a do treino: ViT-Tiny + FFT + EfficientNet-B0.
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Arquitetura IDÊNTICA à da Fase 2 (nomes de módulos precisam casar com o checkpoint).
# ---------- Componentes da Fase 1 (NÃO alterar: precisam casar com os pesos salvos) ----------
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=192):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = ((q @ k.transpose(-2, -1)) * self.scale).softmax(dim=-1)
        attn = self.dropout(attn)
        return self.proj((attn @ v).transpose(1, 2).reshape(B, N, C))


class MLPBlock(nn.Module):
    def __init__(self, embed_dim, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        hidden = int(embed_dim * mlp_ratio)
        self.fc1 = nn.Linear(embed_dim, hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.dropout(self.act(self.fc1(x)))
        return self.dropout(self.fc2(x))


class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = MLPBlock(embed_dim, mlp_ratio, dropout)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class ViTTiny(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 embed_dim=192, depth=12, num_heads=3, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n + 1, embed_dim))
        self.pos_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerEncoderBlock(embed_dim, num_heads, mlp_ratio, dropout)
                                     for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        B = x.shape[0]
        x = torch.cat([self.cls_token.expand(B, -1, -1), self.patch_embed(x)], dim=1)
        x = self.pos_dropout(x + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x)[:, 0]


class FFTBranch(nn.Module):
    def __init__(self, in_channels=3, out_dim=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, stride=2, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, out_dim)

    @staticmethod
    def to_log_magnitude_spectrum(x):
        fft = torch.fft.fftshift(torch.fft.fft2(x, norm="ortho"), dim=(-2, -1))
        return torch.log1p(torch.abs(fft))

    def forward(self, x):
        feat = self.conv(self.to_log_magnitude_spectrum(x))
        return self.fc(self.pool(feat).flatten(1))


# ---------- Novo na Fase 2 ----------
class CNNBranch(nn.Module):
    # EfficientNet-B0 (ImageNet) -> vetor de 1280 dims. Converte a normalização [-1,1] do projeto
    # para a normalização esperada pelos pesos do ImageNet.
    def __init__(self, pretrained=True):
        super().__init__()
        m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        self.features = m.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        x = (x * 0.5 + 0.5 - self.mean) / self.std
        return self.pool(self.features(x)).flatten(1)


class Phase2Detector(nn.Module):
    DIMS = {"vit": 192, "fft": 128, "cnn": 1280}

    def __init__(self, branches=("vit", "fft", "cnn"), fusion="gate", proj_dim=256,
                 pretrained_cnn=True, num_classes=2, dropout=0.2):
        super().__init__()
        self.names = [n for n in ("vit", "fft", "cnn") if n in branches]
        self.fusion = fusion
        mods = {}
        if "vit" in self.names: mods["vit"] = ViTTiny(embed_dim=192)
        if "fft" in self.names: mods["fft"] = FFTBranch(out_dim=128)
        if "cnn" in self.names: mods["cnn"] = CNNBranch(pretrained_cnn)
        self.branches = nn.ModuleDict(mods)
        self.proj = nn.ModuleDict({n: nn.Sequential(nn.LayerNorm(self.DIMS[n]), nn.Linear(self.DIMS[n], proj_dim))
                                   for n in self.names})
        k = len(self.names)
        if fusion == "gate":
            self.gate = nn.Sequential(nn.Linear(k * proj_dim, 64), nn.GELU(), nn.Linear(64, k))
            cls_in = proj_dim
        else:
            self.gate = None
            cls_in = k * proj_dim
        self.classifier = nn.Sequential(nn.LayerNorm(cls_in), nn.Linear(cls_in, 128), nn.GELU(),
                                        nn.Dropout(dropout), nn.Linear(128, num_classes))

    def forward(self, x, return_gate=False):
        feats = [self.proj[n](self.branches[n](x)) for n in self.names]
        gate_w = None
        if self.fusion == "gate":
            gate_w = torch.softmax(self.gate(torch.cat(feats, dim=1)), dim=1)          # (B, k)
            fused = (torch.stack(feats, dim=1) * gate_w.unsqueeze(-1)).sum(dim=1)       # (B, D)
        else:
            fused = torch.cat(feats, dim=1)
        logits = self.classifier(fused)
        return (logits, gate_w) if return_gate else logits

print("Arquitetura definida; pesos ainda nao carregados.")


In [ ]:
# 4. Funcoes de inferencia e apresentacao.
"""Inferencia da demonstracao 3.9; sem treino ou acesso a benchmarks."""

import contextlib
import hashlib
import html
import io
import json
import math
import shutil
from pathlib import Path

import torch
from torchvision import transforms
from PIL import Image, ImageOps, UnidentifiedImageError
from IPython.display import HTML, display


RUN_NAME = 'fase3_9_tgif_sd2fr_v1'
ARCH_SOURCE_SHA256 = '3db23c93de1130c4b55a1b1ad2f3f522005e5e2dd3d15733698f1f2418eb7c7b'
MAX_UPLOAD_BYTES = 20_000_000
MAX_PIXELS = 50_000_000
Image.MAX_IMAGE_PIXELS = MAX_PIXELS // 2  # PIL recusa >50 MP antes de decodificar.


def load_demo_model(run_dir, detector_class, device):
    """Prefere o final calibrado; usa best.pt apenas como demonstracao parcial."""
    run_dir = Path(run_dir)
    complete_path = run_dir / 'training_complete.json'
    final_path = run_dir / 'phase3_final.pt'
    best_path = run_dir / 'best.pt'
    if complete_path.is_file() and final_path.is_file():
        complete = json.loads(complete_path.read_text(encoding='utf-8'))
        if complete.get('status') != 'internal_training_complete':
            raise ValueError('Marcador de treino nao confirma a conclusao')
        source, mode = final_path, 'final_calibrado'
    elif best_path.is_file():
        source, mode = best_path, 'melhor_epoca_sem_calibracao'
    else:
        raise FileNotFoundError('A Fase 3.9 ainda nao tem best.pt nem phase3_final.pt. '
                                'Conclua ao menos uma epoca antes da demonstracao.')

    local = Path('/content/demo_fase3_9_checkpoint.pt')
    shutil.copy2(source, local)
    with local.open('rb') as stream:
        checksum = hashlib.file_digest(stream, 'sha256').hexdigest()
    # Carregar apenas o checkpoint confiavel do proprio grupo, nunca um upload.
    checkpoint = torch.load(local, map_location='cpu', weights_only=False)
    if mode == 'final_calibrado':
        required = {'model_state_dict', 'cfg', 'cfg_phase3', 'temperature', 'threshold'}
        if not isinstance(checkpoint, dict) or not required <= checkpoint.keys():
            raise ValueError('Checkpoint final incompleto')
        config = checkpoint['cfg_phase3']
        architecture = checkpoint['cfg']
        if (tuple(architecture['BRANCHES']) != ('vit', 'fft', 'cnn') or
                architecture['FUSION'] != 'gate' or architecture['PROJ_DIM'] != 256):
            raise ValueError('Arquitetura do checkpoint inesperada')
        weights = checkpoint['model_state_dict']
        temperature = float(checkpoint['temperature'])
        threshold = float(checkpoint['threshold'])
        epoch = config.get('best_epoch', 'final')
    else:
        if not isinstance(checkpoint, dict) or not {'model', 'cfg', 'epoch'} <= checkpoint.keys():
            raise ValueError('Checkpoint best.pt incompleto')
        config = checkpoint['cfg']
        weights = checkpoint['model']
        temperature, threshold = 1.0, None
        epoch = checkpoint['epoch']
    if config.get('RUN_NAME') != RUN_NAME:
        raise ValueError('Checkpoint nao pertence a Fase 3.9')
    if config.get('SOURCE_SHA256') != ARCH_SOURCE_SHA256:
        raise ValueError('Codigo da arquitetura original difere desta demonstracao')
    if not math.isfinite(temperature) or not 0 < temperature < 10:
        raise ValueError('Temperatura invalida')
    if threshold is not None and (not math.isfinite(threshold) or not 0 < threshold < 1):
        raise ValueError('Limiar invalido')

    model = detector_class(branches=('vit', 'fft', 'cnn'), fusion='gate',
                           proj_dim=256, pretrained_cnn=False)
    model.load_state_dict(weights, strict=True)
    model.to(device).eval()
    return {'model': model, 'mode': mode, 'temperature': temperature,
            'threshold': threshold, 'epoch': epoch, 'checkpoint': str(source),
            'sha256': checksum, 'device': device}


_normalize = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
_five_crop = transforms.FiveCrop(224)


def predict_fake_probability(image, loaded):
    """Cinco recortes, media dos logits, temperatura interna quando disponivel."""
    if image.width * image.height > MAX_PIXELS:
        raise ValueError('Imagem acima do limite de 50 megapixels')
    image = ImageOps.exif_transpose(image).convert('RGB')
    if min(image.size) < 224:
        scale = 224 / min(image.size)
        image = image.resize((round(image.width * scale), round(image.height * scale)),
                             Image.Resampling.BICUBIC)
    batch = torch.stack([_normalize(crop) for crop in _five_crop(image)])
    batch = batch.to(loaded['device'])
    amp = (torch.autocast(device_type='cuda', dtype=torch.float16)
           if loaded['device'].type == 'cuda' else contextlib.nullcontext())
    with torch.inference_mode(), amp:
        logits = loaded['model'](batch).float()
    if logits.shape != (5, 2) or not torch.isfinite(logits).all():
        raise ValueError('Saida do modelo invalida')
    return float(torch.softmax(logits.mean(dim=0) / loaded['temperature'], dim=0)[1])


def show_result(name, image, p_fake, loaded):
    p_real = 1 - p_fake
    threshold = loaded['threshold']
    if threshold is None:
        conclusion = 'Checkpoint ainda sem calibracao; sem decisao binaria.'
        note = 'Pontuacao softmax ilustrativa. A temperatura e o limiar ainda nao foram fixados.'
    else:
        conclusion = ('Mais compativel com imagem sintetica' if p_fake >= threshold
                      else 'Mais compativel com imagem real')
        note = 'Estimativa calibrada internamente; nao comprova a origem da imagem.'
    preview = image.copy()
    preview.thumbnail((500, 400))
    display(preview)
    safe_name = html.escape(name)
    display(HTML(
        '<div style="font:16px Arial;max-width:540px;padding:18px;'
        'border:1px solid #ccc;border-radius:12px">'
        f'<b>{safe_name}</b><p style="font-size:20px">{conclusion}</p>'
        f'<p>Sintetica: <b>{p_fake * 100:.1f}%</b></p>'
        f'<div style="background:#eee;height:18px"><div style="background:#a24de0;'
        f'width:{p_fake * 100:.1f}%;height:18px"></div></div>'
        f'<p>Real: <b>{p_real * 100:.1f}%</b></p>'
        f'<div style="background:#eee;height:18px"><div style="background:#218c78;'
        f'width:{p_real * 100:.1f}%;height:18px"></div></div>'
        f'<p style="font-size:13px">{note}</p></div>'
    ))


def analyze_browser_upload(loaded):
    """Seletor de arquivos do navegador Colab; nao funciona em todo VS Code."""
    from google.colab import files
    uploaded = files.upload()
    for name, raw in uploaded.items():
        try:
            if len(raw) > MAX_UPLOAD_BYTES:
                raise ValueError('Arquivo acima de 20 MB')
            with Image.open(io.BytesIO(raw)) as image:
                image.load()
                probability = predict_fake_probability(image, loaded)
                show_result(name, image, probability, loaded)
        except (UnidentifiedImageError, OSError, ValueError) as error:
            print(f'{name}: imagem nao analisada ({error})')


def analyze_drive_folder(loaded, folder):
    """Alternativa ao upload no VS Code: imagens sao lidas da pasta do Drive."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    paths = sorted(p for p in folder.iterdir() if p.is_file() and
                   p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'})
    print('Pasta de entrada:', folder, '| imagens:', len(paths))
    for path in paths:
        try:
            if path.stat().st_size > MAX_UPLOAD_BYTES:
                raise ValueError('Arquivo acima de 20 MB')
            with Image.open(path) as image:
                image.load()
                probability = predict_fake_probability(image, loaded)
                show_result(path.name, image, probability, loaded)
        except (UnidentifiedImageError, OSError, ValueError) as error:
            print(f'{path.name}: imagem nao analisada ({error})')

print("Funcoes prontas; nenhum treino iniciado.")


In [ ]:
# 5. Carregue e verifique estritamente o melhor checkpoint disponivel.
LOADED = load_demo_model(RUN_DIR, Phase2Detector, DEVICE)
print('Checkpoint:', LOADED['checkpoint'])
print('SHA-256:', LOADED['sha256'])
print('Estado:', LOADED['mode'], '| epoca:', LOADED['epoch'])
if LOADED['threshold'] is None:
    print('Treino em andamento: softmax SEM calibracao e sem decisao binaria.')
else:
    print('Temperatura interna:', LOADED['temperature'], '| limiar interno:', LOADED['threshold'])


In [ ]:
# 6. NAVEGADOR COLAB: selecione uma ou mais imagens do seu computador.
# Pelo VS Code, use a celula 7 para evitar o upload do navegador.
analyze_browser_upload(LOADED)


In [ ]:
# 7. DRIVE/VS CODE: coloque JPG, PNG ou WebP na pasta exibida e reexecute.
INPUT_DIR = ROOT / 'apresentacao_fase3_9/entrada'
analyze_drive_folder(LOADED, INPUT_DIR)


## Como explicar o resultado

A barra Sintetica e a estimativa para a classe gerada ou alterada por IA. A barra Real e o complemento. O checkpoint final usa temperatura e limiar definidos em dados internos. Se so houver best.pt, o softmax ainda nao foi calibrado e o notebook nao toma uma decisao binaria. Nenhuma dessas estimativas prova a origem da imagem ou mede a acuracia do detector.
